# Libraries

In [1]:
# Reinstall PyTorch stack with CUDA 12.8 support
!pip uninstall -y torch torchvision torchaudio torchao
!pip install -q \
    torch==2.10.0+cu128 \
    torchvision==0.25.0+cu128 \
    torchaudio==2.10.0+cu128 \
    torchao==0.10.0 \
    --index-url https://download.pytorch.org/whl/cu128

# vLLM
!pip install -q vllm==0.19.0

# Dependencies
!pip install -q "protobuf>=5.26.1,<6.0.0" --break-system-packages

!pip install -q llmcompressor==0.10.0.2
!pip install -q compressed-tensors==0.14.0.1

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 916.9/916.9 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 93.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 55.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.

In [2]:
# Standard library imports
import os
import gc
import json
import time
import random
import shutil
import subprocess

# Third-party libraries
import numpy as np
from vllm import LLM, SamplingParams
from huggingface_hub import login
from transformers import AutoTokenizer, set_seed
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# Deep learning framework
import torch

2026-05-22 12:32:58.131000: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779453178.366926      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779453178.427437      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779453178.990821      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779453178.990898      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779453178.990905      57 computation_placer.cc:177] computation placer alr

# Global Variables

In [3]:
# User & Repository Configuration
USER_NAME = "Abdelrahmanemam01"
EMAIL     = "abdulrahmanemam01@gmail.com"
REPO      = "MahmoudOsama20/EdgeCompress"  # Target repository

# Model Selection
MODEL = "Phi-4-mini-instruct"

# Model & Tokenizer Configuration
MODEL_ID     = "EdgeCompress01/Phi-4-mini-instruct-AWQ-4bit"
TOKENIZER_ID = "microsoft/Phi-4-mini-instruct"

MODEL_NAME           = "Phi-4-mini-instruct"
MODEL_NAME_IN_REPO   = "Phi-4-mini-instruct-AWQ"
COMPRESSION_METHOD   = "AWQ"
MODEL_TYPE           = "Quantization"
#SPARSITY = 70
#PATH = f"Models/{SPARSITY}"

# Inference Prompt (Chat Format)
PROMPT = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Provide a concise explanation of Artificial Intelligence."
    }
]

dummy_prompt = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Give me story of a brave man"
    }
]

# Generation Configuration
MAX_GENERATION_TOKENS = 150
SEED = 42

# Output Configuration
OUTPUT_FILE = "model_metrics.json"

# Functions

In [4]:
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Seeding

In [5]:
set_reproducibility(SEED)

# Huggingface Login

In [6]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# GitHub login

In [7]:
token = UserSecretsClient().get_secret("GIT_TOKEN")
print("Logging into GitHub...")

Logging into GitHub...


# Download Model

In [8]:
"""local_path = snapshot_download(
    repo_id=MODEL_ID,
    allow_patterns=f"{PATH}/*",
    local_dir="/kaggle/working/model"
)"""

'local_path = snapshot_download(\n    repo_id=MODEL_ID,\n    allow_patterns=f"{PATH}/*",\n    local_dir="/kaggle/working/model"\n)'

# Prompt Preprocessing

**DummyPrompt Tokenization**

In [9]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

#Chat Formating
dummy_prompt_text = tokenizer.apply_chat_template(
    dummy_prompt, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
dummy_prompt_token_ids = tokenizer.encode(dummy_prompt_text)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

**RealPrompt Tokenization**

In [10]:
#Chat Formating
prompt_text = tokenizer.apply_chat_template(
    PROMPT, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
prompt_token_ids = tokenizer.encode(prompt_text)

# Initializing vLLM

In [11]:
llm = LLM(
    model=MODEL_ID,
    tokenizer=TOKENIZER_ID,
    dtype="float16",
    max_model_len=256,
    tensor_parallel_size=2,
    gpu_memory_utilization=0.85,
    attention_backend = "TRITON_ATTN",
    disable_log_stats=False,
    enable_prefix_caching = False
)
print("vLLM Initialized Successfully!")


sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_GENERATION_TOKENS,
    min_tokens=MAX_GENERATION_TOKENS,
    seed = SEED
)

ttft_sampling_params = SamplingParams(
    temperature=0.0,
    seed = SEED,
    max_tokens=1,
    ignore_eos=True # Ensures it doesn't accidentally stop early
)

INFO 05-22 12:33:30 [utils.py:233] non-default args: {'tokenizer': 'microsoft/Phi-4-mini-instruct', 'dtype': 'float16', 'max_model_len': 256, 'tensor_parallel_size': 2, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'attention_backend': 'TRITON_ATTN', 'model': 'EdgeCompress01/Phi-4-mini-instruct-AWQ-4bit'}


config.json: 0.00B [00:00, ?B/s]

INFO 05-22 12:33:30 [config.py:446] Replacing legacy 'type' key with 'rope_type'
INFO 05-22 12:33:54 [model.py:549] Resolved architecture: Phi3ForCausalLM
WARNING 05-22 12:33:54 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 05-22 12:33:54 [model.py:1678] Using max model len 256
INFO 05-22 12:33:55 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-22 12:33:55 [vllm.py:790] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/169 [00:00<?, ?B/s]

WARNING 05-22 12:33:57 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


...

(Worker_TP0 pid=272) INFO 05-22 12:34:59 [default_loader.py:384] Loading weights took 1.56 seconds
(Worker_TP0 pid=272) INFO 05-22 12:35:00 [gpu_model_runner.py:4820] Model loading took 1.37 GiB memory and 17.819481 seconds
(Worker_TP0 pid=272) INFO 05-22 12:35:16 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/505a659aff/rank_0_0/backbone for vLLM's torch.compile
(Worker_TP0 pid=272) INFO 05-22 12:35:16 [backends.py:1111] Dynamo bytecode transform time: 14.78 s
(Worker_TP0 pid=272) INFO 05-22 12:35:24 [backends.py:372] Cache the graph of compile range (1, 8192) for later use
(Worker_TP0 pid=272) INFO 05-22 12:35:30 [backends.py:390] Compiling a graph for compile range (1, 8192) takes 13.47 s
(Worker_TP0 pid=272) INFO 05-22 12:35:34 [decorators.py:640] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/b237e7eb599f90ad00782e536906cd64ac3b41aeceab1f05fb996dd164dc0fb1/rank_0_0/model
(Worker_TP0 pid=272) INFO 05-22 12:35:

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 12.14it/s]
Capturing CUDA graphs (decode, FULL):  97%|█████████▋| 34/35 [00:05<00:00,  3.79it/s]

(Worker_TP1 pid=273) INFO 05-22 12:36:08 [custom_all_reduce.py:216] Registering 5590 cuda graph addresses
(Worker_TP0 pid=272) INFO 05-22 12:36:08 [custom_all_reduce.py:216] Registering 5590 cuda graph addresses


Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:06<00:00,  5.00it/s]


(Worker_TP1 pid=273) INFO 05-22 12:36:09 [gpu_worker.py:597] CUDA graph pool memory: 0.47 GiB (actual), 0.49 GiB (estimated), difference: 0.02 GiB (3.7%).
(Worker_TP0 pid=272) INFO 05-22 12:36:09 [gpu_model_runner.py:6046] Graph capturing finished in 13 secs, took 0.47 GiB
(Worker_TP0 pid=272) INFO 05-22 12:36:09 [gpu_worker.py:597] CUDA graph pool memory: 0.47 GiB (actual), 0.49 GiB (estimated), difference: 0.02 GiB (3.7%).
(EngineCore pid=247) INFO 05-22 12:36:09 [core.py:283] init engine (profile, create kv cache, warmup model) took 68.73 seconds
(EngineCore pid=247) INFO 05-22 12:36:11 [vllm.py:790] Asynchronous scheduling is enabled.
vLLM Initialized Successfully!


# Inference

In [12]:
# ── Initialize collectors ──────────────────────────────────────────
ttft_times      = []
latency_times   = []
decode_times    = []
inference_times = []  # prefill + decode

# ── Warm-up ONCE, outside the measurement loop ────────────────────
for _ in range(5):
    llm.generate(
        prompts=[{"prompt_token_ids": dummy_prompt_token_ids}],
        sampling_params=SamplingParams(max_tokens=50, temperature=0.0, seed=SEED),
        use_tqdm=False,
    )

# ── Measurement loop ──────────────────────────────────────────────
N_RUNS = 30
for _ in range(N_RUNS):
    output  = llm.generate(
        prompts=[{"prompt_token_ids": prompt_token_ids}],
        sampling_params=sampling_params,
        use_tqdm=False,
    )
    metrics = output[0].metrics

    ttft      = metrics.first_token_ts - metrics.scheduled_ts
    latency   = metrics.last_token_ts  - metrics.queued_ts
    decode    = metrics.last_token_ts  - metrics.first_token_ts
    inference = ttft + decode                                   # prefill + decode

    ttft_times.append(ttft)
    latency_times.append(latency)
    decode_times.append(decode)
    inference_times.append(inference)

# ── Stats ─────────────────────────────────────────────────────────
ttft_arr, latency_arr, decode_arr, inference_arr = (
    np.array(ttft_times),
    np.array(latency_times),
    np.array(decode_times),
    np.array(inference_times),
)

ttft_mean,      ttft_std      = ttft_arr.mean(),      ttft_arr.std()
latency_mean,   latency_std   = latency_arr.mean(),   latency_arr.std()
inference_mean, inference_std = inference_arr.mean(), inference_arr.std()

# decode throughput: excludes first token, over decode-only window
decode_throughput_arr                           = (MAX_GENERATION_TOKENS - 1) / decode_arr
decode_throughput_mean, decode_throughput_std   = (
    decode_throughput_arr.mean(), decode_throughput_arr.std()
)

# overall throughput: all tokens over e2e latency
overall_throughput_arr                          = MAX_GENERATION_TOKENS / latency_arr
overall_throughput_mean, overall_throughput_std = (
    overall_throughput_arr.mean(), overall_throughput_arr.std()
)

input_tokens           = len(prompt_token_ids)
total_generated_tokens = MAX_GENERATION_TOKENS

INFO 05-22 12:36:23 [loggers.py:259] Engine 000: Avg prompt throughput: 17.1 tokens/s, Avg generation throughput: 88.9 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%
INFO 05-22 12:36:33 [loggers.py:259] Engine 000: Avg prompt throughput: 15.2 tokens/s, Avg generation throughput: 115.0 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
INFO 05-22 12:36:43 [loggers.py:259] Engine 000: Avg prompt throughput: 13.3 tokens/s, Avg generation throughput: 114.6 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%
INFO 05-22 12:36:53 [loggers.py:259] Engine 000: Avg prompt throughput: 15.2 tokens/s, Avg generation throughput: 113.3 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%


In [13]:
print(latency_times)

[1.2934008939998876, 1.2919924860000265, 1.2924762889999784, 1.2976643120000517, 1.2948395110001911, 1.296808098999918, 1.2956208250000145, 1.3018790440000885, 1.30111702399995, 1.3032751810001173, 1.30256341900008, 1.3029275979999966, 1.3033239349999803, 1.3039028860000599, 1.3056958679999298, 1.3067450769999596, 1.3067880470000546, 1.3078720239998347, 1.3171698799999376, 1.3099882340000022, 1.3133730710001146, 1.3144645280001441, 1.3130727669999942, 1.3232793340000626, 1.317842660999986, 1.320647721999876, 1.322492585999953, 1.3254392640001242, 1.3294381799998973, 1.3271262820001084]


In [14]:
print(ttft_times)

[0.03515089999996235, 0.02866522299996177, 0.028432398999939323, 0.03190010000002985, 0.028297839000060776, 0.029155335000041305, 0.027518937999957416, 0.027845058999901084, 0.030012476999900173, 0.03086212599987448, 0.031779913000036686, 0.0311467049998555, 0.02620907300001818, 0.027617065999947954, 0.02962895199993909, 0.02942901799997344, 0.027666423000027862, 0.02762798899993868, 0.03567327300015677, 0.02702617799991458, 0.02924883800005773, 0.028327367999963826, 0.026324111000121775, 0.03296133899993947, 0.025219466999942597, 0.029023393999977998, 0.0274126450001404, 0.02929248100008408, 0.03046666099999129, 0.02818780699999479]


# Results

**Writing in json file**

In [15]:
benchmark_results = {
    "model_name"                        : MODEL_NAME,
    "model_type"                       : MODEL_TYPE,
    "compression_method"               : COMPRESSION_METHOD,
    #"sparsity"                         : SPARSITY,
    "input_token_count"                 : input_tokens,
    "generated_token_count"             : total_generated_tokens,
    "ttft_sec"                          : f"{ttft_mean:.2f} ± {ttft_std:.2f}",
    "inference_sec"                     : f"{inference_mean:.2f} ± {inference_std:.2f}",
    "latency_sec"                       : f"{latency_mean:.2f} ± {latency_std:.2f}",
    "decode_throughput_tokens_per_sec"  : f"{decode_throughput_mean:.2f} ± {decode_throughput_std:.2f}",
    "overall_throughput_tokens_per_sec" : f"{overall_throughput_mean:.2f} ± {overall_throughput_std:.2f}",
}

# Save Results
with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    json.dump(benchmark_results, file, indent=4, ensure_ascii=False)

print(f"[INFO] Metrics successfully saved to '{OUTPUT_FILE}'")

[INFO] Metrics successfully saved to 'model_metrics.json'


**Push Results to GitHub Repository**

In [16]:
# Paths & Repository Setup
target_dir_in_repo = f"Evaluations/InferenceEvaluations/CloudEvaluation/{MODEL}/{MODEL_NAME_IN_REPO}"
source_file = OUTPUT_FILE 
remote_url = f"https://{token}@github.com/{REPO}.git"


# Local temporary directory
current_dir = os.getcwd()
temp_repo_dir = os.path.join(current_dir, "temp_git_repo")


# Clean Previous Runs
if os.path.exists(temp_repo_dir):
    shutil.rmtree(temp_repo_dir)


# Clone Repository
print(f"Cloning repository into: {temp_repo_dir}")
subprocess.run(
    ["git", "clone", remote_url, temp_repo_dir],
    check=True
)


# Prepare Target Directory
full_target_path = os.path.join(temp_repo_dir, target_dir_in_repo)
os.makedirs(full_target_path, exist_ok=True)


# Copy Source File (FIXED)
if os.path.exists(source_file):
    dest_path = os.path.join(full_target_path, os.path.basename(source_file))
    shutil.copy2(source_file, dest_path)
else:
    print(f"Warning: Source file '{source_file}' does not exist.")


# Git Config, Commit & Push
try:
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.email", EMAIL],
        check=True
    )
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.name", USER_NAME],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "add", "."],
        check=True
    )

    commit_msg = f"Added the cloud inference evaluation results to {target_dir_in_repo}"
    subprocess.run(
        ["git", "-C", temp_repo_dir, "commit", "-m", commit_msg],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "push", "origin", "main"],
        check=True
    )

    print(f"Successfully pushed to '{REPO}' → '{target_dir_in_repo}'")

except subprocess.CalledProcessError as error:
    print(f"Git operation failed: {error}")

Cloning repository into: /kaggle/working/temp_git_repo


Cloning into '/kaggle/working/temp_git_repo'...


[main 9ba29e0] Added the cloud inference evaluation results to Evaluations/InferenceEvaluations/CloudEvaluation/Phi-4-mini-instruct/Phi-4-mini-instruct-AWQ
 1 file changed, 12 insertions(+)
 create mode 100644 Evaluations/InferenceEvaluations/CloudEvaluation/Phi-4-mini-instruct/Phi-4-mini-instruct-AWQ/model_metrics.json
Successfully pushed to 'MahmoudOsama20/EdgeCompress' → 'Evaluations/InferenceEvaluations/CloudEvaluation/Phi-4-mini-instruct/Phi-4-mini-instruct-AWQ'


To https://github.com/MahmoudOsama20/EdgeCompress.git
   4a86107..9ba29e0  main -> main
